# easySLM - Customized Fine-Tuning Guide

This notebook mirrors `fine_tune.ipynb` while exposing **all configurable options** in the Python API:
1. `ModelFactory`: custom cache directory (defaults to `~/.easyslm/models`), target device, and dtype.
2. `FineTuningConfig`: all architecture, batching/VRAM, LoRA, optimizer, evaluation, early stopping, oversampling, and MLflow parameters (also loadable via `from_yaml(...)` or `from_dict(...)`).
3. `SLMFineTuner.train(...)`: explicit `val_data_path` and `classes` overrides.
4. Reloading & inference with `SLMBase.from_checkpoint(...)` / `from_best_model(...)` and custom `batch_size`.

In [ ]:
from src import FineTuningConfig, ModelFactory, SLMBase, SLMFineTuner
from src.utils import get_project_root

ROOT = get_project_root()

# 1. Optional: customize ModelFactory (cache_dir defaults to ~/.easyslm/models or $EASYSLM_HOME/models)
factory = ModelFactory(
    cache_dir=None,  # e.g. "./my_models_cache"
    device=None,  # e.g. "cuda" or "cpu" (auto-detected if None)
    torch_dtype=None,  # e.g. torch.bfloat16 / torch.float16 (auto-detected if None)
)

# 2. Configure FineTuningConfig (also supports .from_yaml("configs/training_config.yml", ...) or .from_dict({...}))
config = FineTuningConfig(
    # Architecture & Prompts
    mode="json",  # "cls" (linear head) or "json" (generative)
    pooling_strategy="eos",  # "eos", "mean", or "max" (for mode="cls")
    prompts_config_path=str(ROOT / "configs" / "prompts.yml"),
    output_dir=None,  # defaults to ~/.easyslm/finetuned
    # Batching & VRAM Optimization
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 2 * 4 = 8
    gradient_checkpointing=True,
    # LoRA / PEFT
    use_lora=True,
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    lora_target_modules="all-linear",
    # Optimizer & LR Schedule
    learning_rate=None,  # auto: 2e-4 (LoRA) or 2e-5 (full FT)
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    num_train_epochs=2.0,
    # Evaluation, Splitting, Oversampling & Early Stopping
    val_split_ratio=0.15,  # used only if no val.txt / val_data_path
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",
    save_total_limit=2,
    logging_steps=5,
    oversample_strategy=None,  # None, "max", or "sqrt"
    oversample_max_factor=None,  # e.g. 3.0
    early_stopping_window=4,  # None to disable
    early_stopping_metric="eval_loss",
    seed=42,
    # MLflow Tracking
    use_mlflow=False,  # set True to log to ~/.easyslm/mlflow.db
    mlflow_tracking_uri=None,
    mlflow_experiment_name=None,
    mlflow_config_path=None,
    run_name="agnews-custom-run",
    max_errors_per_class=5,
)

# 3. Instantiate fine-tuner & train (can also pass explicit val_data_path and classes)
tuner = SLMFineTuner(model_name="Qwen/Qwen2.5-0.5B", config=config, factory=factory)
results = tuner.train(
    train_data_path=ROOT / "data" / "agnews_tiny",
    val_data_path=None,
    classes=None,
)

print("Saved checkpoint:", results["checkpoint_dir"])

In [ ]:
import json
from pathlib import Path

# 4. Inspect overall metrics + per-class validation metrics from saved artifacts
eval_m = results["eval_metrics"]
print(
    f"Overall | eval_loss={eval_m.get('eval_loss', 0):.4f} | "
    f"eval_accuracy={eval_m.get('eval_accuracy', 0):.4f} | "
    f"eval_f1_macro={eval_m.get('eval_f1_macro', 0):.4f}\n"
)

metrics_file = Path(results["checkpoint_dir"]) / "tables" / "per_class_metrics.json"
per_class = json.loads(metrics_file.read_text(encoding="utf-8"))

print(f"{'Class':<15} | {'Precision':>9} | {'Recall':>9} | {'F1':>9}")
print("-" * 51)
for cls_name, prec, rec, f1 in zip(
    per_class["class_name"],
    per_class["precision"],
    per_class["recall"],
    per_class["f1_score"],
    strict=True,
):
    print(f"{cls_name:<15} | {prec:>9.4f} | {rec:>9.4f} | {f1:>9.4f}")

In [ ]:
# 5. Reload the exact checkpoint (or use SLMBase.from_best_model) and run batched inference
model = SLMBase.from_checkpoint(
    checkpoint_dir=results["checkpoint_dir"],
    prompts_config_path=None,  # defaults to prompt_config.yml saved inside checkpoint
    factory=factory,
)

custom_texts = [
    "Tech giant unveils next-generation quantum processor for cloud AI workloads.",
    "Central bank holds interest rates steady as quarterly corporate earnings beat forecasts.",
    "National football squad secures dramatic overtime victory in the championship final.",
]

predictions = model.predict_texts(custom_texts, batch_size=4)
for item in predictions:
    print(f"Text:              {item['text']}")
    print(f"Prediction:        {item['predictions']}")
    print(f"Generated Output:  {item.get('raw_output')}\n")